# Benchmark: Portal (GitHub Pages) vs. API (sdicapi)

Este notebook compara o **desempenho** de consultas de dados de emprego obtidas por dois caminhos da biblioteca `sdic_libraries`:

| Caminho | Origem | Funções | Característica |
|---------|--------|---------|----------------|
| **Portal** | Artefatos estáticos JSON publicados no GitHub Pages | `get_relatorio_emprego_*` | Leitura de arquivos pré-processados (CDN) |
| **API** | Serviço dinâmico `sdicapi.dados.ninja` | `get_saldo_emprego_*`, `get_estoque_emprego_*` | Consulta paginada ao vivo no backend |

Para cada caminho medimos o **tempo de resposta (wall-clock)** repetindo a consulta várias vezes em vários estados (UFs) e reportamos:

- menor tempo, maior tempo, **média**, **mediana** e **desvio-padrão**;
- a **distribuição** dos tempos (boxplot e histograma).

> Os dois caminhos não retornam exatamente o mesmo schema, mas representam a mesma informação (saldo CAGED por divisão CNAE e estoque RAIS por divisão CNAE em nível estadual). O foco aqui é o **tempo de obtenção dos dados**, não a igualdade byte-a-byte.

## 1. Configuração

In [ ]:
import time
import statistics
from dataclasses import dataclass, field
from typing import Callable, List, Dict

import pandas as pd
import matplotlib.pyplot as plt

# Funções do PORTAL (GitHub Pages)
from sdic_libraries.dados.emprego.portal import (
    get_relatorio_emprego_saldo_estadual,
    get_relatorio_emprego_estoque_estadual,
)

# Funções da API (sdicapi)
from sdic_libraries.dados.emprego.api import (
    get_saldo_emprego_estadual_mensal,
    get_estoque_emprego_estadual,
)

# ── Parâmetros do benchmark ────────────────────────────────────────────────
UFS = ["SP", "RJ", "MG", "RS", "PR"]   # estados consultados
REPETICOES = 5                          # execuções por (cenário, UF)
AMBIENTE = "homologacao"               # ambiente do portal ('producao' | 'homologacao')

print(f"UFs........: {', '.join(UFS)}")
print(f"Repetições.: {REPETICOES} por UF")
print(f"Total/cenário: {len(UFS) * REPETICOES} chamadas")

## 2. Cronometragem

Cada chamada é executada e cronometrada individualmente. Falhas (rede, UF ausente) são capturadas e **não** entram nas estatísticas de tempo — apenas são contabilizadas como erro.

In [ ]:
@dataclass
class Resultado:
    """Tempos coletados de um cenário (uma combinação caminho + consulta)."""
    cenario: str
    origem: str               # 'Portal' ou 'API'
    tempos: List[float] = field(default_factory=list)   # segundos
    erros: int = 0
    n_linhas: List[int] = field(default_factory=list)


def cronometrar(func: Callable, *args, **kwargs):
    """Executa `func`, retornando (tempo_segundos, n_linhas) ou (None, None) em erro."""
    inicio = time.perf_counter()
    try:
        df = func(*args, **kwargs)
        decorrido = time.perf_counter() - inicio
        n = len(df) if hasattr(df, "__len__") else 0
        return decorrido, n
    except Exception as exc:  # noqa: BLE001 — registra a falha e segue
        print(f"   ⚠️  erro: {type(exc).__name__}: {exc}")
        return None, None


def rodar_cenario(cenario: str, origem: str, func: Callable, ufs: List[str],
                  repeticoes: int, **kwargs_extra) -> Resultado:
    """Roda `func(uf, **kwargs_extra)` para cada UF, `repeticoes` vezes."""
    res = Resultado(cenario=cenario, origem=origem)
    print(f"\n▶ {cenario} [{origem}]")
    for uf in ufs:
        for _ in range(repeticoes):
            t, n = cronometrar(func, uf, **kwargs_extra)
            if t is None:
                res.erros += 1
            else:
                res.tempos.append(t)
                res.n_linhas.append(n)
        print(f"   {uf}: {repeticoes} chamadas concluídas")
    return res

## 3. Cenários comparados

Dois pares equivalentes de consulta:

1. **Saldo estadual (CAGED por divisão)** — Portal `get_relatorio_emprego_saldo_estadual` × API `get_saldo_emprego_estadual_mensal(nivel_cnae='divisao')`
2. **Estoque estadual (RAIS por divisão)** — Portal `get_relatorio_emprego_estoque_estadual` × API `get_estoque_emprego_estadual(nivel_cnae=2)`

In [ ]:
resultados: List[Resultado] = []

# ── Cenário 1: SALDO estadual por divisão ──────────────────────────────────
resultados.append(rodar_cenario(
    "Saldo estadual (divisão)", "Portal",
    get_relatorio_emprego_saldo_estadual, UFS, REPETICOES,
    ambiente=AMBIENTE,
))
resultados.append(rodar_cenario(
    "Saldo estadual (divisão)", "API",
    get_saldo_emprego_estadual_mensal, UFS, REPETICOES,
    nivel_cnae="divisao",
))

# ── Cenário 2: ESTOQUE estadual por divisão ────────────────────────────────
resultados.append(rodar_cenario(
    "Estoque estadual (divisão)", "Portal",
    get_relatorio_emprego_estoque_estadual, UFS, REPETICOES,
    ambiente=AMBIENTE,
))
resultados.append(rodar_cenario(
    "Estoque estadual (divisão)", "API",
    get_estoque_emprego_estadual, UFS, REPETICOES,
    nivel_cnae=2,
))

print("\n✓ Benchmark concluído.")

## 4. Estatísticas resumo

Tempos em **milissegundos**: menor, maior, média, mediana e desvio-padrão.

In [ ]:
def resumo(res: Resultado) -> Dict:
    ms = [t * 1000 for t in res.tempos]
    linha = {
        "Cenário": res.cenario,
        "Origem": res.origem,
        "N": len(ms),
        "Erros": res.erros,
        "Menor (ms)": min(ms) if ms else float("nan"),
        "Maior (ms)": max(ms) if ms else float("nan"),
        "Média (ms)": statistics.mean(ms) if ms else float("nan"),
        "Mediana (ms)": statistics.median(ms) if ms else float("nan"),
        "Desvio (ms)": statistics.pstdev(ms) if len(ms) > 1 else 0.0,
    }
    return linha


df_resumo = pd.DataFrame([resumo(r) for r in resultados])
df_resumo_fmt = df_resumo.copy()
for col in ["Menor (ms)", "Maior (ms)", "Média (ms)", "Mediana (ms)", "Desvio (ms)"]:
    df_resumo_fmt[col] = df_resumo_fmt[col].round(1)
df_resumo_fmt

In [ ]:
# Speedup: quão mais rápido (ou lento) é o Portal vs. a API, por cenário (mediana)
med = df_resumo.pivot_table(index="Cenário", columns="Origem", values="Mediana (ms)")
if {"Portal", "API"}.issubset(med.columns):
    med["Speedup (API/Portal)"] = (med["API"] / med["Portal"]).round(2)
    print("Speedup > 1  ⇒  Portal mais rápido que a API\n")
med.round(1)

## 5. Distribuição dos tempos

In [ ]:
# Tabela longa para os gráficos
linhas = []
for r in resultados:
    for t in r.tempos:
        linhas.append({"Cenário": r.cenario, "Origem": r.origem, "ms": t * 1000})
df_long = pd.DataFrame(linhas)

cenarios = list(dict.fromkeys(df_long["Cenário"]))
cores = {"Portal": "#2a9d8f", "API": "#e76f51"}

fig, axes = plt.subplots(len(cenarios), 2, figsize=(13, 4 * len(cenarios)))
if len(cenarios) == 1:
    axes = axes.reshape(1, 2)

for i, cen in enumerate(cenarios):
    sub = df_long[df_long["Cenário"] == cen]
    origens = [o for o in ("Portal", "API") if o in set(sub["Origem"])]
    dados = [sub[sub["Origem"] == o]["ms"].values for o in origens]

    # Boxplot
    ax_box = axes[i][0]
    bp = ax_box.boxplot(dados, labels=origens, patch_artist=True, showmeans=True)
    for patch, o in zip(bp["boxes"], origens):
        patch.set_facecolor(cores[o])
        patch.set_alpha(0.6)
    ax_box.set_title(f"{cen} — distribuição")
    ax_box.set_ylabel("tempo (ms)")
    ax_box.grid(True, axis="y", alpha=0.3)

    # Histograma
    ax_hist = axes[i][1]
    for o in origens:
        vals = sub[sub["Origem"] == o]["ms"].values
        ax_hist.hist(vals, bins=12, alpha=0.6, label=o, color=cores[o])
    ax_hist.set_title(f"{cen} — histograma")
    ax_hist.set_xlabel("tempo (ms)")
    ax_hist.set_ylabel("frequência")
    ax_hist.legend()
    ax_hist.grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Comparativo geral das médias (barras com barra de erro = desvio-padrão)
fig, ax = plt.subplots(figsize=(11, 5))
largura = 0.35
x = range(len(cenarios))

for j, origem in enumerate(("Portal", "API")):
    medias, desvios = [], []
    for cen in cenarios:
        linha = df_resumo[(df_resumo["Cenário"] == cen) & (df_resumo["Origem"] == origem)]
        medias.append(linha["Média (ms)"].values[0] if not linha.empty else 0)
        desvios.append(linha["Desvio (ms)"].values[0] if not linha.empty else 0)
    pos = [xi + (j - 0.5) * largura for xi in x]
    ax.bar(pos, medias, largura, yerr=desvios, capsize=4,
           label=origem, color=cores[origem], alpha=0.8)

ax.set_xticks(list(x))
ax.set_xticklabels(cenarios, rotation=10)
ax.set_ylabel("tempo médio (ms)")
ax.set_title("Tempo médio por cenário — Portal vs. API (barra de erro = desvio-padrão)")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Conclusão

A célula abaixo gera um resumo textual automático a partir dos números coletados.

In [ ]:
print("RESUMO DO BENCHMARK")
print("=" * 60)
for cen in cenarios:
    print(f"\n• {cen}")
    for origem in ("Portal", "API"):
        linha = df_resumo[(df_resumo["Cenário"] == cen) & (df_resumo["Origem"] == origem)]
        if linha.empty:
            continue
        r = linha.iloc[0]
        print(f"   {origem:>6}: média={r['Média (ms)']:.0f}ms  "
              f"mediana={r['Mediana (ms)']:.0f}ms  "
              f"min={r['Menor (ms)']:.0f}ms  max={r['Maior (ms)']:.0f}ms  "
              f"(N={int(r['N'])}, erros={int(r['Erros'])})")
    if {"Portal", "API"}.issubset(set(med.columns)) and cen in med.index:
        sp = med.loc[cen, "Speedup (API/Portal)"]
        mais_rapido = "Portal" if sp > 1 else "API"
        print(f"   → {mais_rapido} é ~{max(sp, 1/sp):.1f}x mais rápido (mediana)")